In [ ]:
import random

# Constants
BOARD_WIDTH = 18
BOARD_HEIGHT = 9
OBSTACLES = [(1,1),(1,2),(1,3),(1,5),(1,6),(1,7),(2,1),(2,7),(3,3),(3,5),(4,0),(4,1),(4,3),(4,5),(4,7),(4,8),
                      (6,1),(6,3),(6,4),(6,5),(6,7),(7,1),(7,3),(7,5),(7,7),(8,1),(8,3),(8,7),(9,1),(9,3),(9,7),(10,1),
                      (10,3),(10,5),(10,7),(11,1),(11,3),(11,4),(11,5),(11,7),(13,0),(13,1),(13,3),(13,5),(13,7),(13,8),
                      (14,3),(14,5),(15,1),(15,7),(16,1),(16,2),(16,3),(16,5),(16,6),(16,7)] # These can be adjusted
PACMAN_START = (0, 0)
GHOSTS_START = [(8,8), (4,4)]
MOVES = { 'S':(0,0),'U': (0, 1), 'R': (1, 0), 'L': (-1, 0),'D': (0, -1)}
MAX_DEPTH = 9
PACMAN_MOVES = []

# Game state
class GameState:
    def __init__(self, pacman, ghosts, points, score):
        self.pacman = pacman
        self.ghosts = ghosts
        self.points = points
        self.score = score

    def is_terminal(self):
        return not self.points or any(ghost == self.pacman for ghost in self.ghosts)

    def utility(self):

        if self.is_terminal():
            if self.pacman in self.ghosts:
                return float("-inf")
            else:
                return float("inf")
        else:
            distance1 = abs(self.pacman[0]-self.ghosts[0][0]) +abs(self.pacman[1]-self.ghosts[0][1])
            distance2 = abs(self.pacman[0]-self.ghosts[1][0]) +abs(self.pacman[1]-self.ghosts[1][1])

            k=0
            if min(distance1, distance2) > 3:
                k=100
            elif min(distance1, distance2)< 2:
                k=-1000
            else:
                k =-100

            return self.score + k



# Minimax algorithm
def minimax(state, depth, agent):
    if depth == 0 or state.is_terminal():
        return state.utility()
    if agent==0:
        max_eval = float("-inf")
        for move in MOVES.values():
            if is_valid_position((state.pacman[0] + move[0], state.pacman[1] + move[1])):
                child = get_child_state(state, move, True)
                eval = minimax(child, depth-1, 1)
                max_eval = max(max_eval, eval)
        return max_eval
    elif agent ==1:
        min_eval = float("inf")
        for move in MOVES.values():
            if is_valid_position((state.ghosts[0][0] + move[0], state.ghosts[0][1] + move[1])):
                child = get_child_state(state, move, False, 0)
                eval = minimax(child, depth-1, 2)
                min_eval = min(min_eval, eval)
        return min_eval
    elif agent ==2:
        min_eval = float("inf")
        for move in MOVES.values():
            if is_valid_position((state.ghosts[1][0] + move[0], state.ghosts[1][1] + move[1])):
                child = get_child_state(state, move, False, 1)
                eval = minimax(child, depth-1, 0)
                min_eval = min(min_eval, eval)
        return min_eval

def get_child_state(state, move, is_pacman, ghost_index=None):
    if is_pacman:
        new_position = (state.pacman[0] + move[0], state.pacman[1] + move[1])
        if new_position in OBSTACLES or not is_valid_position(new_position):
            new_position = state.pacman
        points = state.points - {new_position} if new_position in state.points else state.points
        return GameState(new_position, state.ghosts, points, state.score + (200 if new_position in state.points else -10) )
    else:
        new_ghosts = list(state.ghosts)
        new_position = (state.ghosts[ghost_index][0] + move[0], state.ghosts[ghost_index][1] + move[1])
        if new_position in OBSTACLES or not is_valid_position(new_position):
            new_position = state.ghosts[ghost_index]
        new_ghosts[ghost_index] = new_position
        return GameState(state.pacman, tuple(new_ghosts), state.points, state.score)

def get_best_child_state(state, move):
    new_position = (state.pacman[0] + move[0], state.pacman[1] + move[1])
    if new_position in OBSTACLES or not is_valid_position(new_position):
        new_position = state.pacman
    points = state.points - {new_position} if new_position in state.points else state.points
    return GameState(new_position, state.ghosts, points, state.score + (10 if new_position in state.points else -1) )

def is_valid_position(position):
    row, col = position
    return 0 <= row < BOARD_WIDTH and 0 <= col < BOARD_HEIGHT and position not in OBSTACLES
def is_pacman_stuck(state):
    pass

def play_game():
    points = {(row, col) for row in range(BOARD_WIDTH) for col in range(BOARD_HEIGHT) if (row, col) not in OBSTACLES}
    state = GameState(PACMAN_START, GHOSTS_START, points, 0)
    while not state.is_terminal():
        best_move = None
        best_eval = float("-inf")
        moves = [(0,0),(1,0),(-1,0),(0,1),(0,-1)]
        for move in moves:
            if is_valid_position((state.pacman[0] + move[0], state.pacman[1] + move[1])):
                if len(PACMAN_MOVES)>3 :
                    if not( (state.pacman[0] + move[0], state.pacman[1] + move[1]) == PACMAN_MOVES[len(PACMAN_MOVES)-2] and PACMAN_MOVES[len(PACMAN_MOVES)-1] == PACMAN_MOVES[len(PACMAN_MOVES)-3]) and not ((state.pacman[0] + move[0], state.pacman[1] + move[1]) == PACMAN_MOVES[len(PACMAN_MOVES)-1] and PACMAN_MOVES[len(PACMAN_MOVES)-1] == PACMAN_MOVES[len(PACMAN_MOVES)-2]):
                        child = get_child_state(state, move, True)
                        eval = minimax(child, MAX_DEPTH-1, 1)
                        if eval >= best_eval:
                            best_eval = eval
                            best_move = move
                else:
                    child = get_child_state(state, move, True)
                    eval = minimax(child, MAX_DEPTH-1, 1)
                    if eval >= best_eval:
                        best_eval = eval
                        best_move = move

        state = get_best_child_state(state, best_move)

        PACMAN_MOVES.append(state.pacman)

        print(f"Pacman is at {state.pacman}, Score: {state.score}")
        for i, _ in enumerate(state.ghosts):
            random_move = random.choice(list(MOVES.values()))
            state = get_child_state(state, random_move, False, i)
            print(f"Ghost {i+1} is at {state.ghosts[i]}")
        map=""
        for i in reversed(range(0,9)):
            for j in range(0,18):
                if (j,i) in OBSTACLES:
                    map = map + "#"

                elif (j,i) == state.pacman:
                    map = map + "p"

                elif (j,i) == state.ghosts[0]:
                    map = map + "g"

                elif (j,i) == state.ghosts[1]:
                    map = map + "s"

                elif (j,i) in state.points:
                    map = map + "."

                else:
                    map = map + " "
                map=map+" "
            map = map +"\n"

        print(map)
        # print(state.score)
        if not state.points:
            print("Pacman wins!")
            break
        elif state.pacman in state.ghosts:
            print("Pacman is caught by a ghost!")
            break

play_game()



Streaming output truncated to the last 5000 lines.
  #                             #   
  #   # #   # #     # #   # #   #   
            # s     . #             
  #   # #   # # # # # #   # #   #   
  #                             #   
  # # p #   # # # # # #   #   # # g 
        #                 #         

Pacman is at (3, 2), Score: -2654
Ghost 1 is at (17, 2)
Ghost 2 is at (7, 4)
        #   . .           #         
  # #   #   # # # # # #   #   # #   
  #                             #   
  #   # #   # #     # #   # #   #   
            # s     . #             
  #   # #   # # # # # #   # #   #   
  #   p                         # g 
  # #   #   # # # # # #   #   # #   
        #                 #         

Pacman is at (3, 1), Score: -2655
Ghost 1 is at (17, 2)
Ghost 2 is at (8, 4)
        #   . .           #         
  # #   #   # # # # # #   #   # #   
  #                             #   
  #   # #   # #     # #   # #   #   
            #   s   . #             
  #   # #   # # 